# 02 - Model Training (YOLO, transfer learning)

**What is YOLO?** "You Only Look Once" is a family of detectors that predict boxes and class
probabilities for the whole image in a **single forward pass**, which makes it fast enough for
real-time use (unlike older two-stage detectors that first propose regions and then classify them).

**Transfer learning:** we start from weights pretrained on COCO (80 classes, ~118k images). Early
layers already know edges, textures and shapes, so we only need to adapt the model to our 4 classes.
This needs far less data, time and compute than training from scratch.

**Lightweight model (YOLO11n, "nano"):** ~2.6M parameters; trains on Colab/consumer GPUs and even
CPU, and runs fast in the web app. Larger variants (s/m/l) are more accurate but slower.

In [ ]:
import sys
from pathlib import Path

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from common import *
print("Project root:", ROOT)

In [ ]:
import shutil
import torch
from ultralytics import YOLO
import ultralytics
print("Ultralytics:", ultralytics.__version__, "| Torch:", torch.__version__)

DATA_YAML = DATASET_DIR / "dataset.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"{DATA_YAML} missing - run notebook 01 first")

## GPU detection and hardware-dependent settings

In [ ]:
cuda = torch.cuda.is_available()
print("CUDA available:", cuda)
print(torch.cuda.get_device_name(0) if cuda else "CPU")

if cuda:
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    device = 0
    cfg = dict(imgsz=640, batch=16 if vram >= 8 else 8, epochs=30)
else:
    print("No GPU: training on CPU with a smaller image size, batch and epochs (slower, less accurate).")
    print("Tip: use Google Colab (Runtime > Change runtime type > GPU) for the full setting.")
    device = "cpu"
    cfg = dict(imgsz=416, batch=8, epochs=10)
print("Settings:", cfg)

## Train
`patience=10` stops early if validation does not improve for 10 epochs. `seed=42` for reproducibility.
If CUDA runs out of memory the cell retries automatically with half the batch size and image size.

In [ ]:
TRAIN_DIR = OUTPUTS_DIR / "training"       # absolute path -> no accidental runs/detect/... nesting
RUN_NAME = "yolo_kitti_vehicles"

def train(batch, imgsz, epochs):
    model = YOLO("yolo11n.pt")  # pretrained weights are auto-downloaded on first use
    model.train(data=str(DATA_YAML), epochs=epochs, imgsz=imgsz, batch=batch,
                patience=10, device=device, workers=0, seed=42, deterministic=True,
                project=str(TRAIN_DIR), name=RUN_NAME, exist_ok=True)
    return model

batch, imgsz, epochs = cfg["batch"], cfg["imgsz"], cfg["epochs"]
while True:
    try:
        model = train(batch, imgsz, epochs)
        break
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if batch <= 2:
            raise RuntimeError("Out of memory even at batch=2: use CPU or a smaller model")
        batch, imgsz = batch // 2, max(320, imgsz // 2 if batch <= 4 else imgsz)
        print(f"CUDA OOM -> retrying with batch={batch}, imgsz={imgsz}")

save_dir = Path(model.trainer.save_dir)
print("Run saved in:", save_dir)

## Save best weights to `models/best.pt`

In [ ]:
best = save_dir / "weights" / "best.pt"
if not best.exists():
    raise FileNotFoundError(f"Expected weights at {best}")
MODELS_DIR.mkdir(exist_ok=True)
shutil.copy2(best, BEST_MODEL)
print("Saved:", BEST_MODEL, f"({BEST_MODEL.stat().st_size / 1e6:.1f} MB)")

## Training metrics - what they mean
| Metric | Meaning |
|---|---|
| **box_loss** | How far predicted boxes are from ground truth (CIoU-based). Lower = better localisation. |
| **cls_loss** | Classification error (right animal?). Lower = better. |
| **dfl_loss** | Distribution Focal Loss: refines box edge positions. Lower = sharper boxes. |
| **Precision** | Of the boxes predicted, the fraction that are correct (few false alarms). |
| **Recall** | Of the real objects, the fraction found (few misses). |
| **mAP@50** | Mean over classes of average precision at IoU >= 0.50 (lenient overlap). |
| **mAP@50-95** | mAP averaged over IoU 0.50 to 0.95 (strict, main COCO metric). |

Loss falling on train but not on validation indicates **overfitting**. No particular score is guaranteed - it depends on the run.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

res = pd.read_csv(save_dir / "results.csv")
res.columns = [c.strip() for c in res.columns]
display(res.tail(3))

groups = {
    "Box loss": ["train/box_loss", "val/box_loss"],
    "Classification loss": ["train/cls_loss", "val/cls_loss"],
    "DFL loss": ["train/dfl_loss", "val/dfl_loss"],
    "Precision / Recall": ["metrics/precision(B)", "metrics/recall(B)"],
    "mAP": ["metrics/mAP50(B)", "metrics/mAP50-95(B)"],
}
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (title, cols) in zip(axes.ravel(), groups.items()):
    for c in cols:
        ax.plot(res["epoch"], res[c], marker="o", ms=3, label=c.split("/")[-1])
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
axes.ravel()[-1].axis("off")
plt.tight_layout()
(OUTPUTS_DIR / "plots").mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUTS_DIR / "plots" / "training_curves.png", dpi=150)
plt.show()

best_row = res.loc[res["metrics/mAP50-95(B)"].idxmax()]
print(f"Best epoch (by mAP50-95 on the VALIDATION set during training): {int(best_row['epoch'])}")
print(best_row[["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]])

Next: `03_model_evaluation.ipynb`.